# Building a Simple LLM with PyTorch: Setup and Data Preprocessing

This notebook walks through the initial setup and data preprocessing steps for building a simple Large Language Model (LLM) using PyTorch. We'll cover:

1. Setting up the environment
2. Downloading and exploring the WikiText-2 dataset
3. Implementing tokenization
4. Creating data loaders for training

Let's get started!

## 1. Environment Setup

First, let's ensure we have all required packages installed. If you haven't already, install the requirements using:

```bash
pip install -r requirements.txt
```

Now let's import the necessary packages:

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import torch
from datasets import load_dataset
from src.data import DataModule

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Loading the WikiText-2 Dataset

We'll use the WikiText-2 dataset, which contains high-quality Wikipedia articles. The dataset is a good choice for this tutorial because:
- It's manageable in size (~2M words)
- Contains well-written, coherent text
- Has built-in train/validation/test splits

Let's load and explore the dataset:

In [ ]:
# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Print basic information
print("Dataset splits:")
for split in dataset.keys():
    print(f"- {split}: {len(dataset[split])} examples")

# Look at a sample from training data
print("\nSample text from training data:")
print(dataset["train"][0]["text"][:500])

## 3. Data Preprocessing

Our `DataModule` class handles all the data preprocessing steps. Let's create an instance and examine how it works:

In [ ]:
# Initialize data module
data_module = DataModule(
    seq_length=512,  # Maximum sequence length
    stride=256,      # Stride for sliding window
    batch_size=32,   # Batch size for training
    vocab_size=30000 # Size of vocabulary
)

# Prepare data (this will download the dataset and train the tokenizer)
data_module.prepare_data()

### 3.1 Examining the Tokenizer

Let's look at how our trained tokenizer processes text:

In [ ]:
# Get tokenizer from data module
tokenizer = data_module.tokenizer

# Sample text to tokenize
sample_text = "Hello, this is a sample text for our language model!"

# Encode text
encoding = tokenizer.encode(sample_text)

print("Original text:", sample_text)
print("\nTokens:", encoding.tokens)
print("\nToken IDs:", encoding.ids)

### 3.2 Examining the Dataset

Now let's look at how our dataset processes and prepares the data for training:

In [ ]:
# Get a batch from the training dataloader
train_dataloader = data_module.train_dataloader()
batch = next(iter(train_dataloader))

print("Batch contents:")
for key, value in batch.items():
    print(f"- {key}: shape {value.shape}, dtype {value.dtype}")

# Decode a sample sequence
sample_idx = 0
input_sequence = batch["input_ids"][sample_idx]
decoded_text = tokenizer.decode(input_sequence.tolist())

print("\nSample decoded text:")
print(decoded_text[:200], "...")

## 4. Preparing for Training

Let's verify that our data module has properly prepared everything we need for training:

In [ ]:
# Check dataloaders
train_dataloader = data_module.train_dataloader()
val_dataloader = data_module.val_dataloader()
test_dataloader = data_module.test_dataloader()

print("Dataset sizes:")
print(f"- Training: {len(train_dataloader.dataset)} samples")
print(f"- Validation: {len(val_dataloader.dataset)} samples")
print(f"- Test: {len(test_dataloader.dataset)} samples")

print(f"\nBatch size: {train_dataloader.batch_size}")
print(f"Number of training batches: {len(train_dataloader)}")

## Next Steps

We've successfully:
1. Set up our development environment
2. Downloaded and explored the WikiText-2 dataset
3. Trained a tokenizer on our data
4. Created data loaders for training

In the next notebook, we'll implement and train our language model using this preprocessed data.